In [1]:
# ---------------------------
# 0) Installs
# ---------------------------
!pip install -q sentence-transformers faiss-cpu torch torchvision torchaudio tqdm scikit-learn pandas numpy requests


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 83.0 MB/s eta 0:00:00


In [2]:
# ---------------------------
# 1) Imports & config
# ---------------------------
import os, sys, sqlite3, json, random, time, math, re
from tqdm import tqdm
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer
import requests
import torch, torch.nn as nn, torch.optim as optim
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from collections import defaultdict, Counter


In [3]:
# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [4]:
# ---------------------------
# 2) User-editable paths & keys
# ---------------------------
DB_PATH = "/content/rdguru.db"   # <<-- upload your sqlite file here / adjust path
MODEL_SAVE = "/content/ddqn_rdgurupp.pt"

# Groq API config - optional. Put your key if you plan to call Groq for reasoning or XDE.
GROQ_API_KEY = os.getenv("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")  # or set string directly, e.g. "sk-..."
GROQ_MODEL = "llama-3.1-8b-instant"  # example; adjust to your Groq model
USE_GROQ = bool(GROQ_API_KEY)

In [5]:
# Seed
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)


In [7]:
# ---------------------------
# 3) Load SQLite tables
# ---------------------------
assert os.path.exists(DB_PATH), f"DB file not found at {DB_PATH}. Upload it to Colab and update DB_PATH."
conn = sqlite3.connect(DB_PATH)
df_summary = pd.read_sql_query("SELECT * FROM SUMMARY", conn)
df_pheno   = pd.read_sql_query("SELECT * FROM PHENOTYPES", conn)
df_cases   = pd.read_sql_query("SELECT * FROM CASE_TEXT_RECORD", conn)

print("Loaded tables:", df_summary.shape, df_pheno.shape, df_cases.shape)

# Basic clean: ensure expected columns exist (handle lower/upper-case variants)
def col_if_exists(df, names):
    for n in names:
        if n in df.columns:
            return n
    return None

# try best-effort column names
col_orpha = col_if_exists(df_summary, ["orpha","ORPHA","orpha_id","orphaid"])
col_pmid  = col_if_exists(df_summary, ["pmid","PMID","pmid_id"])
col_dname = col_if_exists(df_summary, ["disease_name","disease","diseaseName","DISEASE_NAME"])
print("Summary cols used:", col_orpha, col_pmid, col_dname)


Loaded tables: (476, 5) (238, 5) (102, 5)
Summary cols used: orpha pmid disease_name


In [8]:
!pip install pronto

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.3/63.3 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 18.0 MB/s eta 0:00:00


In [10]:
# ---------------------------
# 4) Build disease -> HPO profile (with .obo ontology mapping)
# ---------------------------
# We'll use the official HPO ontology to expand keyword→HPO mapping dynamically.

from pronto import Ontology
import json

hp_ontology = Ontology("hp.obo")
hp_regex = re.compile(r"HP:\d{7}")

# Helper to extract HP codes from text
def extract_hpo_from_text(s):
    if not isinstance(s, str):
        return []
    return list(set(hp_regex.findall(s)))

# Build keyword-to-HPO dictionary dynamically
# (We’ll index all HPO terms by their names and synonyms)
hpo_keyword_map = {}
for term in hp_ontology.terms():
    if term.name:
        hpo_keyword_map[term.name.lower()] = term.id
    for syn in term.synonyms:
        hpo_keyword_map[syn.description.lower()] = term.id

print(f"✅ Loaded {len(hpo_keyword_map)} keyword→HPO mappings from hp.obo")

disease_profiles = {}   # d_key -> set(HPO)
disease_texts = {}      # d_key -> concatenated text for embedding
d_key_list = []

# Compose disease keys
for idx, row in df_summary.iterrows():
    orpha = row.get(col_orpha, None)
    pmid = row.get(col_pmid, None)
    dname = row.get(col_dname, f"disease_{idx}")
    if orpha and str(orpha).strip() != "":
        key = f"ORPHA:{str(orpha)}"
    elif pmid and str(pmid).strip() != "":
        key = f"PMID:{str(pmid)}"
    else:
        key = f"SUMIDX:{idx}"
    text = str(dname) + " " + str(row.get("file_name","") or "")
    try:
        if pmid and pmid != "":
            matches = df_cases[df_cases['pmid'] == pmid]['text_report'].fillna("").tolist()
            if matches:
                text += " " + " ".join(matches[:3])
    except Exception:
        pass
    disease_texts[key] = text
    disease_profiles[key] = set()
    d_key_list.append(key)

# From CASE_TEXT_RECORD
for _, row in df_cases.iterrows():
    pmid = row.get('pmid', None)
    dname = str(row.get('disease_name', '') or "")
    txt = str(row.get('text_report', '') or "")
    found = extract_hpo_from_text(txt)
    if pmid and len(found)>0:
        key = f"PMID:{pmid}"
        disease_profiles.setdefault(key, set()).update(found)
        disease_texts.setdefault(key, txt)
    else:
        for sidx, srow in df_summary.iterrows():
            sname = str(srow.get(col_dname,"") or "")
            if sname and sname.lower() in dname.lower():
                orpha = srow.get(col_orpha, None)
                pm = srow.get(col_pmid, None)
                k = f"ORPHA:{orpha}" if orpha else (f"PMID:{pm}" if pm else f"SUMIDX:{sidx}")
                disease_profiles.setdefault(k, set()).update(found)

# From PHENOTYPES table
for _, row in df_pheno.iterrows():
    raw = row.get("hpo_terms", None)
    hpos = []
    if isinstance(raw, str):
        try:
            parsed = json.loads(raw)
            if isinstance(parsed, list): hpos = parsed
        except Exception:
            hpos = extract_hpo_from_text(raw)
    elif isinstance(raw, list):
        hpos = raw
    identifier = str(row.get("identifier", "") or "")
    if identifier:
        matched = df_summary[df_summary[col_pmid].astype(str) == identifier] if col_pmid else pd.DataFrame()
        if not matched.empty:
            row0 = matched.iloc[0]
            orpha = row0.get(col_orpha, None)
            pm = row0.get(col_pmid, None)
            key = f"ORPHA:{orpha}" if orpha else (f"PMID:{pm}" if pm else None)
            if key:
                disease_profiles.setdefault(key, set()).update(hpos)
                disease_texts.setdefault(key, "")

# Fallback: detect HPOs from text + ontology keywords
for k, txt in list(disease_texts.items()):
    if len(disease_profiles.get(k, [])) == 0:
        found_hpos = extract_hpo_from_text(txt)
        if not found_hpos:
            detected = []
            for kw, hp_id in hpo_keyword_map.items():
                if kw in txt.lower():
                    detected.append(hp_id)
            found_hpos = detected
        if found_hpos:
            disease_profiles[k].update(found_hpos)

empty_profiles = [k for k,v in disease_profiles.items() if len(v)==0]
print("Total disease keys:", len(disease_profiles), "Empty profiles:", len(empty_profiles))

# If still empty, generate minimal keyword-based HPO fallback (dynamic, from .obo keywords)
for k in empty_profiles:
    text = disease_texts.get(k,"").lower()
    added = set()
    for term, hp_id in hpo_keyword_map.items():
        if term in text:
            added.add(hp_id)
    if added:
        disease_profiles[k].update(added)

empty_profiles2 = [k for k,v in disease_profiles.items() if len(v)==0]
print("Remaining empty disease profiles:", len(empty_profiles2))

disease_keys = list(disease_profiles.keys())
print("Final disease count:", len(disease_keys))


/tmp/ipython-input-3877153054.py:9: UnicodeWarning: unsound encoding, assuming ISO-8859-1 (73% confidence)
  hp_ontology = Ontology("hp.obo")


✅ Loaded 42798 keyword→HPO mappings from hp.obo
Total disease keys: 92 Empty profiles: 3
Remaining empty disease profiles: 3
Final disease count: 92


In [11]:
# ---------------------------
# 5) Precompute per-disease HPO frequency (for pseudo-PheLR)
# ---------------------------
# We'll compute frequency of each HPO across disease profiles to approximate prevalence weight.
hpo_counts = Counter()
for k, s in disease_profiles.items():
    for hp in s:
        hpo_counts[hp] += 1

# disease->list(HPO) and freq map
disease_hpo_list = {k: sorted(list(v)) for k,v in disease_profiles.items()}
print("Example disease -> HPO counts (sample):", list(disease_hpo_list.items())[:3])


Example disease -> HPO counts (sample): [('ORPHA:77298', ['HP:0000528', 'HP:0000568', 'HP:0001658', 'HP:0002032', 'HP:0031353']), ('ORPHA:1340', ['HP:0031353']), ('ORPHA:634', ['HP:0031353'])]


In [12]:
# ---------------------------
# 6) Build embeddings & FAISS index (disease textual embeddings)
# ---------------------------
print("Loading SBERT model (this may take a moment)...")
emb_model = SentenceTransformer("all-MiniLM-L6-v2")
d_keys = list(disease_texts.keys())
d_texts = [disease_texts[k] if disease_texts[k] else d_keys[i] for i,k in enumerate(d_keys)]
d_embs = emb_model.encode(d_texts, convert_to_numpy=True, show_progress_bar=True)
d_embs = d_embs.astype('float32')
# Normalize for cosine
emb_norms = np.linalg.norm(d_embs, axis=1, keepdims=True) + 1e-12
d_embs_norm = d_embs / emb_norms

# FAISS index (for RAG retrieval from your CASE_TEXT_RECORD docs)
# We'll also index the CASE_TEXT_RECORD text docs for retrieval in Flow A
case_docs = df_cases["text_report"].fillna("").astype(str).tolist()
if len(case_docs)==0:
    case_docs = d_texts  # fallback
case_embs = emb_model.encode(case_docs, convert_to_numpy=True, show_progress_bar=True).astype('float32')
faiss.normalize_L2(case_embs)
case_idx = faiss.IndexFlatIP(case_embs.shape[1])
case_idx.add(case_embs)
print("FAISS case index built. Cases:", len(case_docs))

# Also build disease embedding FAISS for phenotype matching if needed
faiss.normalize_L2(d_embs)
disease_idx = faiss.IndexFlatIP(d_embs.shape[1])
disease_idx.add(d_embs)

Loading SBERT model (this may take a moment)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

FAISS case index built. Cases: 102


In [13]:
# ---------------------------
# 7) Synthetic patient generation (N synthetic cases)
# ---------------------------
N_SYN = 10000
print(f"Generating {N_SYN} synthetic patients from disease profiles...")

synthetic_cases = []
d_keys_nonempty = [k for k in d_keys if len(disease_hpo_list.get(k,[]))>0]
if len(d_keys_nonempty)==0:
    # fallback: micro-generate random HP groups from fallback_map
    for i in range(N_SYN):
        hpos = random.sample(list(fallback_map.values())[0], k=1)
        synthetic_cases.append({"hpos": hpos, "ground": d_keys[0]})
else:
    # sample disease proportional to how many HPO terms they have (or uniform)
    for i in range(N_SYN):
        # pick a disease
        k = random.choice(d_keys_nonempty)
        hplist = disease_hpo_list[k]
        # sample # symptoms 3..min(8,len(hplist))
        if len(hplist) <= 3:
            sample_count = len(hplist)
        else:
            sample_count = random.randint(3, min(8, len(hplist)))
        # sample with possible noise: replace one HPO with similar or random occasionally
        sampled = random.sample(hplist, sample_count)
        # add noise: with small prob add unrelated HPs
        if random.random() < 0.08:
            # add a random HP from global set
            all_hpos = list(hpo_counts.keys())
            sampled.append(random.choice(all_hpos))
        # random demographic (not used much but stored)
        age = random.choice(["P1Y","P5Y","P17Y","P25Y","P40Y"])
        sex = random.choice(["MALE","FEMALE"])
        synthetic_cases.append({"hpos": sampled, "ground": k, "age": age, "sex": sex})

print("Synthetic cases generated:", len(synthetic_cases))

# Combine synthetic with real cases from df_pheno (if they have ground mapping)
# Build real cases list with ground mapping: try to map PHENOTYPES to disease_keys
real_cases = []
for _, r in df_pheno.iterrows():
    raw = r.get("hpo_terms", None)
    hpos = []
    if isinstance(raw, str):
        try:
            parsed = json.loads(raw)
            if isinstance(parsed, list):
                hpos = parsed
            else:
                hpos = extract_hpo_from_text(raw)
        except Exception:
            hpos = extract_hpo_from_text(raw)
    elif isinstance(raw, list):
        hpos = raw
    else:
        hpos = []
    # ground mapping heuristics: try to use 'identifier' -> summary pmid -> disease key
    ground = None
    identifier = str(r.get("identifier","") or "")
    if identifier:
        matched = df_summary[df_summary[col_pmid].astype(str)==identifier] if col_pmid else pd.DataFrame()
        if not matched.empty:
            orpha = matched.iloc[0].get(col_orpha, None)
            pm = matched.iloc[0].get(col_pmid, None)
            ground = f"ORPHA:{orpha}" if orpha and f"ORPHA:{orpha}" in disease_profiles else (f"PMID:{pm}" if pm and f"PMID:{pm}" in disease_profiles else None)
    if hpos and ground:
        real_cases.append({"hpos":hpos, "ground":ground})

print("Real mapped cases for training (with ground):", len(real_cases))

# Training set: combine synthetic + real (we'll mostly train on synthetic, validate on real)
train_cases = synthetic_cases + real_cases
random.shuffle(train_cases)

# Build evaluation set: hold out some real cases for validation
# Use the real_cases list as evaluation (since they are actual cases)
eval_cases = real_cases.copy()
print("Training cases:", len(train_cases), "Evaluation (real) cases:", len(eval_cases))


Generating 10000 synthetic patients from disease profiles...
Synthetic cases generated: 10000
Real mapped cases for training (with ground): 201
Training cases: 10201 Evaluation (real) cases: 201


In [14]:
# ---------------------------
# 8) Implement three scorers
#    - PheLR-like: score based on HPO presence & IDF-like weighting
#    - Phenotype matcher: Jaccard or embedding cosine
#    - Groq LLM reasoning: optional real call or placeholder merging
# ---------------------------

# Precompute disease HPO sets & idf
all_hpos = list(hpo_counts.keys())
idf = {hp: math.log1p(len(disease_keys)/(1 + hpo_counts.get(hp,0))) for hp in all_hpos}

def scorer_phelr(hpos):
    """Approximate likelihood score: sum of idf-weighted matches per disease"""
    scores = {}
    hset = set(hpos)
    for i, k in enumerate(d_keys):
        dhpos = set(disease_hpo_list.get(k, []))
        if not dhpos:
            scores[k] = 0.0
            continue
        # weighted overlap
        overlap = hset.intersection(dhpos)
        s = sum(idf.get(h, 1.0) for h in overlap)
        # normalize by disease profile length
        s = s / (len(dhpos)**0.5)
        scores[k] = s
    # sort descending
    ranked = sorted(scores.items(), key=lambda x: -x[1])
    return ranked

def scorer_matcher(hpos):
    """Embedding-based similarity: average HPO embedding -> cosine to disease embeddings"""
    if len(hpos)==0:
        # fallback: all zero
        return [(k, 0.0) for k in d_keys]
    # embed HPO terms (we use ID strings or map to names if available)
    h_embs = emb_model.encode(hpos, convert_to_numpy=True)
    v = h_embs.mean(axis=0).reshape(1,-1).astype('float32')
    faiss.normalize_L2(v)
    # use cosine to disease embeddings (which are normalized already)
    sims = (d_embs @ v.T).squeeze()  # dot product since normalized
    ranked_idx = np.argsort(-sims)
    return [(d_keys[i], float(sims[i])) for i in ranked_idx]

def scorer_groq(hpos):
    """Groq call to request ranked list (optional). If GROQ not available, fallback to a noisy merge"""
    if USE_GROQ:
        # Build prompt: request top 20 ORPHA/PMID keys by HPO list
        prompt = f"Patient HPOs: {hpos}\nReturn a JSON list of up to 20 candidate diseases in the format " + \
                 "\"[{\\\"id\\\":\\\"ORPHA:xxxxx\\\", \\\"name\\\":\\\"Name\\\", \\\"score\\\":0.87}, ...]\".\nOnly use ORPHA: or PMID: ids."
        # minimal Groq client (adjust url and payload to your Groq provider)
        headers = {"Authorization": f"Bearer {GROQ_API_KEY}"}
        body = {"model": GROQ_MODEL, "messages":[{"role":"user","content":prompt}], "temperature":0.0}
        try:
            r = requests.post("https://api.groq.com/openai/v1/chat/completions", json=body, headers=headers, timeout=30)
            text = r.json()["choices"][0]["message"]["content"]
            # try parse JSON in text
            parsed = json.loads(text)
            ranks = []
            for it in parsed:
                kid = it.get("id")
                score = float(it.get("score", 0.0))
                if kid and kid in d_keys:
                    ranks.append((kid, score))
            # pad missing diseases with low scores
            existing = set([k for k,_ in ranks])
            for k in d_keys:
                if k not in existing:
                    ranks.append((k, 0.0))
            return ranks
        except Exception as e:
            print("Groq call failed:", e)
            # fallback to placeholder below
    # Fallback noisy merge between the two scorers
    p = scorer_phelr(hpos)[:200]
    m = scorer_matcher(hpos)[:200]
    # average normalized ranks
    rank_score = {}
    for rank, (k, s) in enumerate(p):
        rank_score.setdefault(k, []).append(1.0/(rank+1))
    for rank, (k, s) in enumerate(m):
        rank_score.setdefault(k, []).append(1.0/(rank+1))
    merged = []
    for k, vals in rank_score.items():
        merged.append((k, sum(vals)/len(vals)))
    merged_sorted = sorted(merged, key=lambda x: -x[1])
    # ensure full ranking
    existing = [k for k,_ in merged_sorted]
    for k in d_keys:
        if k not in existing:
            merged_sorted.append((k, 0.0))
    return merged_sorted

# Test scorers on a sample HPO set
sample_hpos = list(next(iter(disease_hpo_list.values())) )[:3] if disease_hpo_list else ["HP:0003678"]
print("Sample HPOs:", sample_hpos)
print("PheLR top 3:", scorer_phelr(sample_hpos)[:3])
print("Matcher top 3:", scorer_matcher(sample_hpos)[:3])
print("Groq (fallback) top 3:", scorer_groq(sample_hpos)[:3])


Sample HPOs: ['HP:0000528', 'HP:0000568', 'HP:0001658']
PheLR top 3: [('ORPHA:77298', 4.327869827981645), ('ORPHA:2131', 1.873274144418343), ('ORPHA:2604', 1.3246048505396384)]
Matcher top 3: [('ORPHA:125', 0.31252574920654297), ('ORPHA:58', 0.26398301124572754), ('ORPHA:3463', 0.2630741000175476)]
Groq (fallback) top 3: [('ORPHA:125', 0.5227272727272727), ('ORPHA:77298', 0.5092592592592593), ('ORPHA:58', 0.2916666666666667)]


In [15]:
# ---------------------------
# 9) DDQN Environment (fusion) and networks
# ---------------------------
# We'll model the environment where action chooses which source to take next (0=PheLR,1=Matcher,2=Groq)
# State includes: compressed patient embedding (PCA), topN availability flags or scores, fused list length
TOPN = 10
K_FUSED = 10

# Prepare PCA/scaler for patient embedding compression
# Fit PCA on disease embeddings (and optionally on many synthetic patient embeddings)
all_emb_pool = np.vstack([d_embs])
scaler = StandardScaler().fit(all_emb_pool)
pca = PCA(n_components=64, random_state=SEED)
pca.fit(scaler.transform(all_emb_pool))

def hp_list_to_state_vector(hpos):
    if len(hpos)==0:
        vec = np.zeros(emb_model.get_sentence_embedding_dimension(), dtype=np.float32)
    else:
        v = emb_model.encode(hpos, convert_to_numpy=True).mean(axis=0)
        vec = v.astype(np.float32)
    vs = scaler.transform(vec.reshape(1,-1))
    vp = pca.transform(vs)[0]
    return vp  # 64-d

# Fusion environment
class FusionEnv:
    def __init__(self, case, topn=TOPN, k_fused=K_FUSED):
        self.case = case
        self.hpos = case["hpos"]
        self.ground = case.get("ground", None)
        self.patient_vec = hp_list_to_state_vector(self.hpos)
        # compute each source ranked lists (topn)
        self.p1 = [k for k,_ in scorer_phelr(self.hpos)[:topn]]
        self.p2 = [k for k,_ in scorer_matcher(self.hpos)[:topn]]
        self.p3 = [k for k,_ in scorer_groq(self.hpos)[:topn]]
        self.lists = [self.p1, self.p2, self.p3]
        self.ptrs = [0,0,0]
        self.fused = []
        self.k = k_fused
        self.steps = 0
        self.done = False

    def get_state(self):
        # patient vector 64-d
        v = self.patient_vec  # 64
        # topN availability scores: for each source/slot,  value proportional to remaining count
        scores = []
        for lst, ptr in zip(self.lists, self.ptrs):
            for i in range(TOPN):
                scores.append(1.0 if i+ptr < len(lst) else 0.0)
        # fused_len one-hot up to K_FUSED
        fused_len = [1.0 if i < len(self.fused) else 0.0 for i in range(self.k)]
        state = np.concatenate([v, np.array(scores, dtype=np.float32), np.array(fused_len, dtype=np.float32)], axis=0)
        return state

    def step(self, action):
        assert action in [0,1,2]
        src = action
        reward = 0.0
        self.steps += 1
        # choose next candidate from selected source (skip duplicates)
        chosen = None
        while self.ptrs[src] < len(self.lists[src]):
            cand = self.lists[src][self.ptrs[src]]
            self.ptrs[src] += 1
            if cand not in [x["orpha"] for x in self.fused]:
                chosen = cand
                self.fused.append({"orpha": cand, "source": src})
                break
        # reward shaping
        if self.ground:
            # if chosen is ground
            if chosen == self.ground:
                reward += 3.0
            # reward if ground is present in fused top-k
            topk = [x["orpha"] for x in self.fused][:5]
            if self.ground in topk:
                reward += 1.0
        # small step penalty
        reward -= 0.01
        # done cond
        if len(self.fused) >= self.k or self.steps >= self.k*3:
            self.done = True
        return self.get_state(), reward, self.done, {}

    def reset(self):
        self.ptrs = [0,0,0]
        self.fused = []
        self.steps = 0
        self.done = False
        return self.get_state()

# Replay buffer
class ReplayBuffer:
    def __init__(self, size=50000):
        self.size = size
        self.buf = []
        self.pos = 0
    def add(self, transition):
        if len(self.buf) < self.size:
            self.buf.append(transition)
        else:
            self.buf[self.pos] = transition
        self.pos = (self.pos + 1) % self.size
    def sample(self, n):
        batch = random.sample(self.buf, n)
        s,a,r,s2,d = zip(*batch)
        return np.stack(s), np.array(a), np.array(r, dtype=np.float32), np.stack(s2), np.array(d, dtype=np.uint8)
    def __len__(self): return len(self.buf)

# Q-network
class MLPQ(nn.Module):
    def __init__(self, state_dim, action_dim=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 512),
            nn.ReLU(),
            nn.Linear(512,256),
            nn.ReLU(),
            nn.Linear(256, action_dim)
        )
    def forward(self, x):
        return self.net(x)

# Create nets
STATE_DIM = hp_list_to_state_vector(["HP:0003678"]).shape[0] + (3*TOPN) + K_FUSED
online_net = MLPQ(STATE_DIM, 3).to(device)
target_net = MLPQ(STATE_DIM, 3).to(device)
target_net.load_state_dict(online_net.state_dict())
optimizer = optim.Adam(online_net.parameters(), lr=1e-4)
replay = ReplayBuffer(size=50000)

# Epsilon schedule
EPS_START = 0.6; EPS_END = 0.05; EPS_DECAY = 50000.0
def get_epsilon(step):
    return EPS_END + (EPS_START-EPS_END)*math.exp(-1.0*step/EPS_DECAY)


In [16]:
# ---------------------------
# 10) Warmup and training loop
# ---------------------------
NUM_TRAIN_STEPS = 50000  # you can increase for better performance
WARMUP_EPISODES = 2000
BATCH_SIZE = 128
GAMMA = 0.9
TARGET_UPDATE = 1000

print("Warmup replay with random policy... (episodes)", WARMUP_EPISODES)
# Warmup episodes
for _ in tqdm(range(WARMUP_EPISODES)):
    c = random.choice(train_cases)
    env = FusionEnv(c)
    s = env.reset()
    done = False
    while not done:
        a = random.randrange(3)
        s2, r, done, _ = env.step(a)
        replay.add((s, a, r, s2, done))
        s = s2

print("Warmup done. Replay size:", len(replay))

step = 0
pbar = tqdm(range(NUM_TRAIN_STEPS))
for _ in pbar:
    step += 1
    # Collectone episode
    c = random.choice(train_cases)
    env = FusionEnv(c)
    s = env.reset()
    done = False
    while not done:
        eps = get_epsilon(step)
        if random.random() < eps:
            a = random.randrange(3)
        else:
            with torch.no_grad():
                st = torch.tensor(s, dtype=torch.float32).unsqueeze(0).to(device)
                q = online_net(st)
                a = int(torch.argmax(q, dim=1).item())
        s2, r, done, _ = env.step(a)
        replay.add((s,a,r,s2,done))
        s = s2

    # Training step
    if len(replay) >= BATCH_SIZE:
        s_b, a_b, r_b, s2_b, done_b = replay.sample(BATCH_SIZE)
        s_b = torch.tensor(s_b, dtype=torch.float32).to(device)
        a_b = torch.tensor(a_b, dtype=torch.long).to(device)
        r_b = torch.tensor(r_b, dtype=torch.float32).to(device)
        s2_b = torch.tensor(s2_b, dtype=torch.float32).to(device)
        done_b = torch.tensor(done_b, dtype=torch.float32).to(device)

        qvals = online_net(s_b)
        q_s_a = qvals.gather(1, a_b.unsqueeze(1)).squeeze(1)

        # Double DQN target
        with torch.no_grad():
            a_max = torch.argmax(online_net(s2_b), dim=1)
            q_target_vals = target_net(s2_b).gather(1, a_max.unsqueeze(1)).squeeze(1)
            y = r_b + (1.0 - done_b) * GAMMA * q_target_vals

        loss = nn.MSELoss()(q_s_a, y)
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(online_net.parameters(), 5.0)
        optimizer.step()

    if step % TARGET_UPDATE == 0:
        target_net.load_state_dict(online_net.state_dict())

    if step % 2000 == 0:
        # quick evaluation on a sample of eval_cases
        def eval_policy(net, cases, n_eval=100):
            top1=top3=top5=0; total=0
            for c in random.sample(cases, min(len(cases), n_eval)):
                env = FusionEnv(c)
                s = env.reset(); done=False
                while not done:
                    st = torch.tensor(s, dtype=torch.float32).unsqueeze(0).to(device)
                    with torch.no_grad():
                        q = net(st)
                        a = int(torch.argmax(q, dim=1).item())
                    s, r, done, _ = env.step(a)
                fused = [x["orpha"] for x in env.fused]
                gt = c.get("ground")
                total += 1
                if gt is None: continue
                if gt in fused[:1]: top1 += 1
                if gt in fused[:3]: top3 += 1
                if gt in fused[:5]: top5 += 1
            return {"top1": top1/total if total>0 else 0.0, "top3": top3/total if total>0 else 0.0, "top5": top5/total if total>0 else 0.0}
        metrics = eval_policy(online_net, eval_cases, n_eval=100)
        pbar.set_description(f"step {step} loss {loss.item():.4f} top5 {metrics['top5']:.3f}")

# Save model
torch.save({"online_state": online_net.state_dict()}, MODEL_SAVE)
print("Saved DDQN to", MODEL_SAVE)


Warmup replay with random policy... (episodes) 2000


100%|██████████| 2000/2000 [00:42<00:00, 46.68it/s]


Warmup done. Replay size: 20000


step 50000 loss 0.2068 top5 1.000: 100%|██████████| 50000/50000 [23:29<00:00, 35.47it/s]

Saved DDQN to /content/ddqn_rdgurupp.pt


In [17]:
# ---------------------------
# 11) Load trained model for inference
# ---------------------------
ckpt = torch.load(MODEL_SAVE, map_location=device)
trained_net = MLPQ(STATE_DIM, 3).to(device)
trained_net.load_state_dict(ckpt["online_state"])
trained_net.eval()
print("DDQN loaded for inference.")

# Utility to run greedy inference
def predict_ddqn(hpos, topk=5):
    case = {"hpos": hpos, "ground": None}
    env = FusionEnv(case)
    s = env.reset(); done=False
    while not done:
        st = torch.tensor(s, dtype=torch.float32).unsqueeze(0).to(device)
        with torch.no_grad():
            q = trained_net(st)
            a = int(torch.argmax(q, dim=1).item())
        s, r, done, _ = env.step(a)
    fused = [x["orpha"] for x in env.fused]
    # Map ORPHA/PMID keys to human names if possible
    def name_for(k):
        # try to find disease name in df_summary
        if k.startswith("ORPHA:") or k.startswith("PMID:"):
            # search df_summary
            v = None
            if k.startswith("ORPHA:"):
                code = k.split("ORPHA:")[1]
                m = df_summary[df_summary[col_orpha].astype(str)==str(code)] if col_orpha else pd.DataFrame()
                if not m.empty: v = m.iloc[0].get(col_dname)
            elif k.startswith("PMID:"):
                pm = k.split("PMID:")[1]
                m = df_summary[df_summary[col_pmid].astype(str)==str(pm)] if col_pmid else pd.DataFrame()
                if not m.empty: v = m.iloc[0].get(col_dname)
            return v if v else k
        return k
    return [(fid, name_for(fid)) for fid in fused[:topk]]


DDQN loaded for inference.


In [18]:
# ---------------------------
# 12) RAG + AKR + Phenotype extractor + XDE glue
# ---------------------------
# Minimal Groq client wrapper (as before); uses your GROQ_API_KEY if set.
class GroqClient:
    def __init__(self, api_key, model="mixtral-8x7b"):
        self.api_key = api_key
        self.model = model
        self.url = "https://api.groq.com/openai/v1/chat/completions"
    def generate(self, prompt, temperature=0.0, max_tokens=512):
        if not self.api_key:
            # fallback: return a short template
            return "[Groq not configured] " + prompt[:300]
        headers = {"Authorization": f"Bearer {self.api_key}"}
        body = {"model": self.model, "messages":[{"role":"user","content":prompt}], "temperature": temperature, "max_tokens": max_tokens}
        r = requests.post(self.url, json=body, headers=headers, timeout=30)
        return r.json()["choices"][0]["message"]["content"]

groq_client = GroqClient(GROQ_API_KEY, GROQ_MODEL)


def select_sources(query):
    q = query.lower()
    if any(k in q for k in ["gene","mutation","variant"]): return ["OMIM"]
    if any(k in q for k in ["patient","symptom","hp:","hpo","weakness","ptosis","seizure"]): return ["Orphadata","HPO"]
    if any(k in q for k in ["treatment","therapy","management"]): return ["GARD"]
    return ["Orphadata"]

def retrieve_docs(query, k=4):
    # simple FAISS retrieval from case_docs
    q_emb = emb_model.encode([query], convert_to_numpy=True).astype('float32')
    faiss.normalize_L2(q_emb)
    D,I = case_idx.search(q_emb, k)
    res = []
    for i in I[0]:
        if i < len(case_docs):
            res.append({"doc": case_docs[i], "idx": i})
    return res

def extract_hpo_terms(text):
    # if text already contains HP codes, return them; else attempt to find token keywords->HP mapping
    found = extract_hpo_from_text(text)
    if found: return found
    # fallback mapping using small dictionary
    mapping = {"muscle weakness":"HP:0003678","ptosis":"HP:0000508","elevated CK":"HP:0003233","seizure":"HP:0001250"}
    res = []
    for k,v in mapping.items():
        if k in text.lower():
            res.append(v)
    return res

def build_xde_note(top_candidates, hpos, retrieved_docs):
    # top_candidates: list of (orpha, name) or keys
    # retrieved_docs: list of dicts with "doc" and "idx"
    # Compose structured prompt for Groq
    sources_text = "\n\n".join([f"[DOC {i}] {d['doc'][:800]}" for i,d in enumerate(retrieved_docs)])
    candidates_text = "\n".join([f"{i+1}. {c[0]} - {c[1]}" for i,c in enumerate(top_candidates[:5])])
    prompt = f"""
You are a clinical assistant. Patient HPOs: {hpos}.
Top candidates (with brief name):
{candidates_text}

Supporting source excerpts:
{sources_text}

Task: Write a concise 3-paragraph clinician note:
1) short diagnostic suggestion sentence,
2) reasons linking patient HPOs to the top candidate (include inline citations like [DOC 0]),
3) recommended next diagnostic tests.

Keep it professional and concise.
"""
    out = groq_client.generate(prompt, temperature=0.2, max_tokens=400)
    return out

In [19]:
# ---------------------------
# 13) Integrate into two-flow pipeline
# ---------------------------
def classify_query(query):
    q = query.lower()
    if any(k in q for k in ["patient","year old","yr old","hp:","hpo","symptom","weakness","ptosis","seizure"]):
        return "case"
    return "info"

def rdguru_pp_pipeline(query):
    flow = classify_query(query)
    if flow == "info":
        # Flow A: RAG
        sources = select_sources(query)
        ctxs = retrieve_docs(query, k=4)
        context = "\n\n".join([c["doc"] for c in ctxs])
        prompt = f"You are a clinical knowledge assistant. Use only the following verified excerpts to answer the question.\n\n{context}\n\nQ: {query}\nA:"
        answer = groq_client.generate(prompt, temperature=0.0, max_tokens=300)
        return {"flow":"RAG", "answer": answer, "sources": ctxs}
    else:
        # Flow B: Medical Consultation
        hpos = extract_hpo_terms(query)
        # Retrieve supporting docs for XDE
        ctxs = retrieve_docs(" ".join(hpos if hpos else [query]), k=4)
        # scorers (optional: call real Groq for reasoning here)
        # but for fusion we directly use DDQN inference
        fused = predict_ddqn(hpos, topk=10)
        # Map to human names and produce XDE note
        top_named = [(k, (df_summary[(df_summary[col_orpha].astype(str)==k.split('ORPHA:')[-1])][col_dname].iloc[0] if (k.startswith("ORPHA:") and (df_summary[col_orpha].astype(str)==k.split('ORPHA:')[-1]).any()) else k)) for k,_ in fused]
        xde_note = build_xde_note(top_named, hpos, ctxs)
        return {"flow":"MedicalConsult", "hpos": hpos, "top_candidates": top_named[:5], "xde_note": xde_note, "sources": ctxs}


In [20]:
# ---------------------------
# 14) Test both flows
# ---------------------------
print("=== Test: Flow A (Info) ===")
resA = rdguru_pp_pipeline("What causes Gaucher disease?")
print(resA["answer"][:800])

print("\n=== Test: Flow B (Case) ===")
case_query = "8 year old female with muscle weakness, ptosis, and elevated CK"
resB = rdguru_pp_pipeline(case_query)
print("Extracted HPOs:", resB["hpos"])
print("Top candidates:", resB["top_candidates"])
print("XDE note sample:\n", resB["xde_note"][:800])

# ---------------------------
# 15) Evaluate on held-out real cases (Top-1/3/5)
# ---------------------------
def evaluate_on_cases(cases):
    top1=top3=top5=0; total=0
    for c in cases:
        if not c.get("ground"): continue
        preds = predict_ddqn(c["hpos"], topk=10)
        pred_keys = [p[0] for p in preds]
        gt = c["ground"]
        total += 1
        if gt in pred_keys[:1]: top1 += 1
        if gt in pred_keys[:3]: top3 += 1
        if gt in pred_keys[:5]: top5 += 1
    return {"top1": top1/total if total else None, "top3": top3/total if total else None, "top5": top5/total if total else None, "n": total}

eval_metrics = evaluate_on_cases(eval_cases)
print("Evaluation on real cases:", eval_metrics)

# ---------------------------
# 16) Conversation logging (SQLite)
# ---------------------------
def init_log_table():
    with sqlite3.connect(DB_PATH) as c:
        c.execute("""
        CREATE TABLE IF NOT EXISTS CONVERSATION_LOG(
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            ts REAL,
            flow TEXT,
            query TEXT,
            hpos TEXT,
            top_candidates TEXT,
            xde_note TEXT
        )
        """)
init_log_table()

def log_conversation(query, result):
    with sqlite3.connect(DB_PATH) as c:
        hpos = json.dumps(result.get("hpos", []))
        tops = json.dumps(result.get("top_candidates", []))
        note = result.get("xde_note", "")
        c.execute("INSERT INTO CONVERSATION_LOG(ts, flow, query, hpos, top_candidates, xde_note) VALUES(?,?,?,?,?,?)",
                  (time.time(), result.get("flow"), query, hpos, tops, note))
    print("Logged conversation to CONVERSATION_LOG.")

# Example log
log_conversation(case_query, resB)

print("Notebook run completed. You now have a trained DDQN, RAG+AKR in place, and both flows wired together.")
print("Model saved at:", MODEL_SAVE)

# End of notebook


=== Test: Flow A (Info) ===
[Groq not configured] You are a clinical knowledge assistant. Use only the following verified excerpts to answer the question.

A 46 year old female of Northern European descent (patient 1 in reference [7] and patient 27 in Additional file 1) was diagnosed with ABL at age 11, when she presented with right ptosis. Medical

=== Test: Flow B (Case) ===
Extracted HPOs: ['HP:0003678', 'HP:0000508']
Top candidates: [('ORPHA:98855', 'Autosomal recessive Emery-Dreifuss muscular dystrophy'), ('ORPHA:125', 'Bloom syndrome'), ('ORPHA:169186', 'Autosomal recessive centronuclear myopathy'), ('ORPHA:58', 'Alexander disease'), ('ORPHA:3463', 'Wolfram syndrome')]
XDE note sample:
 [Groq not configured] 
You are a clinical assistant. Patient HPOs: ['HP:0003678', 'HP:0000508'].
Top candidates (with brief name):
1. ORPHA:98855 - Autosomal recessive Emery-Dreifuss muscular dystrophy
2. ORPHA:125 - Bloom syndrome
3. ORPHA:169186 - Autosomal recessive centronuclear myopathy
4. OR